# IPL Crunch '26 - Data Analytics Challenge

**Author:** [Your Name Here]  
**Tools:** Python, Pandas, Matplotlib, Seaborn


In [ ]:
# 1. Upload your data files
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from IPython.display import Image, display

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

OUTPUT_DIR = "/content/ipl_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Upload matches.csv and deliveries.csv")
uploaded = files.upload()

file_names = list(uploaded.keys())
print(f"Uploaded files: {file_names}")

matches_file = [f for f in file_names if "match" in f.lower()][0]
deliveries_file = [f for f in file_names if "deliver" in f.lower()][0]
print(f"Loading: {matches_file}, {deliveries_file}")

matches = pd.read_csv("/content/" + matches_file)
deliveries = pd.read_csv("/content/" + deliveries_file)

print(f"Matches: {len(matches)}")
print(f"Deliveries: {len(deliveries):,}")
print(f"Seasons: {sorted(matches['season_year'].dropna().unique().astype(int))}")

In [ ]:
matches.head(3)

In [ ]:
deliveries.head(3)

In [ ]:
# 2. Feature Engineering
def get_phase(over):
    if over <= 6: return "Powerplay (1-6)"
    elif over <= 15: return "Middle (7-15)"
    else: return "Death (16-20)"

deliveries["over"] = deliveries["ball"].astype(float).apply(lambda x: int(x))
deliveries["phase"] = deliveries["over"].apply(get_phase)
deliveries["total_runs"] = deliveries["runs_off_bat"] + deliveries["extras"]
deliveries["is_wicket"] = (
    deliveries["wicket_type"].notna() & (deliveries["wicket_type"] != "run out")
).astype(int)
deliveries["batting_team_won"] = (deliveries["batting_team"] == deliveries["winner"]).astype(int)
matches["toss_winner_won_match"] = (matches["toss_winner"] == matches["winner"]).astype(int)
print("Done")

## Question 1: Do teams that win the toss actually win more matches?

In [ ]:
toss_win_rate = matches["toss_winner_won_match"].mean() * 100
print(f"Toss winner wins: {toss_win_rate:.1f}% of matches")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ['Toss Winner Won', 'Toss Winner Lost']
values = [matches['toss_winner_won_match'].sum(), len(matches) - matches['toss_winner_won_match'].sum()]
axes[0].pie(values, labels=labels, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Toss Winner -> Match Winner?')

decision_groups = matches.groupby('toss_decision')['toss_winner_won_match'].agg(['count', 'mean'])
decision_groups['mean'] *= 100
sns.barplot(data=decision_groups.reset_index(), x='toss_decision', y='mean', hue='toss_decision',
            palette={'bat': '#3498db', 'field': '#f39c12'}, ax=axes[1], legend=False)
axes[1].set_ylabel('Win %'); axes[1].set_ylim(0, 100); axes[1].set_title('Win % by Toss Decision')
for i, row in decision_groups.reset_index().iterrows():
    axes[1].text(i, row['mean'] + 1, f"{row['mean']:.1f}%", ha='center', fontweight='bold')

season_toss = matches.groupby('season_year')['toss_winner_won_match'].mean() * 100
axes[2].plot(season_toss.index.astype(int), season_toss.values, marker='o', color='#8e44ad', linewidth=2)
axes[2].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Season'); axes[2].set_ylabel('Toss Winner Win %')
axes[2].set_title('Toss Advantage Over Seasons'); axes[2].set_ylim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_toss_vs_win.png'), dpi=150, bbox_inches='tight')
plt.show()

## Question 2: Which phase impacts victory the most?

In [ ]:
phase_stats = deliveries.groupby(["batting_team_won", "phase"]).agg(
    balls=("ball", "count"), runs=("total_runs", "sum"), wickets=("is_wicket", "sum")
).reset_index()
phase_stats["run_rate"] = (phase_stats["runs"] / phase_stats["balls"]) * 6
phase_stats["wickets_per_over"] = (phase_stats["wickets"] / phase_stats["balls"]) * 6

pivot_rr = phase_stats.pivot_table(index="phase", columns="batting_team_won", values="run_rate")
pivot_rr.columns = ["Losing Side", "Winning Side"]
print(pivot_rr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pivot_rr[['Winning Side', 'Losing Side']].plot(kind='bar', ax=axes[0],
    color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_ylabel('Run Rate'); axes[0].set_title('Run Rate by Phase')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.2f', fontsize=9)

pivot_wk = phase_stats.pivot_table(index="phase", columns="batting_team_won", values="wickets_per_over")
pivot_wk.columns = ["Losing Side", "Winning Side"]
pivot_wk[['Winning Side', 'Losing Side']].plot(kind='bar', ax=axes[1],
    color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_ylabel('Wickets per Over'); axes[1].set_title('Wickets per Over by Phase')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.2f', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_phase_impact.png'), dpi=150, bbox_inches='tight')
plt.show()

## Question 3: Top Batters Across Seasons

In [ ]:
batter_stats = deliveries.groupby("striker").agg(
    balls_faced=("ball", "count"),
    total_runs=("runs_off_bat", "sum"),
    dismissals=("is_wicket", lambda x: (deliveries.loc[x.index, "player_dismissed"] == deliveries.loc[x.index, "striker"]).sum())
).reset_index()
batter_stats = batter_stats[batter_stats["balls_faced"] >= 500].copy()
batter_stats["average"] = batter_stats["total_runs"] / batter_stats["dismissals"].replace(0, np.nan)
batter_stats["strike_rate"] = (batter_stats["total_runs"] / batter_stats["balls_faced"]) * 100
print(batter_stats.nlargest(10, "total_runs")[["striker", "total_runs", "average", "strike_rate"]].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, col, title, pal in zip(axes, ['total_runs','average','strike_rate'],
    ['Top 15 Runs','Top 15 Average','Top 15 Strike Rate'], ['viridis','magma','plasma']):
    d = batter_stats.nlargest(15, col)
    sns.barplot(data=d, y='striker', x=col, ax=ax, palette=pal, hue='striker', legend=False)
    ax.set_title(title); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_top_batters.png'), dpi=150, bbox_inches='tight')
plt.show()

season_batters = deliveries.groupby(['season_year', 'striker']).agg(runs=('runs_off_bat','sum'), balls=('ball','count')).reset_index()
season_batters = season_batters[season_batters['balls'] >= 100]
top = season_batters.loc[season_batters.groupby('season_year')['runs'].idxmax()]
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=top, x='season_year', y='runs', hue='striker', ax=ax, dodge=False, legend=False)
for _, r in top.iterrows(): ax.text(int(r['season_year']), r['runs']+10, r['striker'], ha='center', fontsize=8, rotation=45)
ax.set_title('Highest Run-Scorer Each Season'); ax.set_xlabel('Season'); ax.set_ylabel('Runs')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03b_top_batter_per_season.png'), dpi=150, bbox_inches='tight')
plt.show()

## Question 4: Top Bowlers Across Seasons

In [ ]:
bowler_stats = deliveries.groupby("bowler").agg(
    balls_bowled=("ball","count"), runs_conceded=("total_runs","sum"), wickets=("is_wicket","sum")
).reset_index()
bowler_stats = bowler_stats[bowler_stats["balls_bowled"] >= 300].copy()
bowler_stats["economy"] = (bowler_stats["runs_conceded"] / bowler_stats["balls_bowled"]) * 6
bowler_stats["average"] = bowler_stats["runs_conceded"] / bowler_stats["wickets"].replace(0, np.nan)
print(bowler_stats.nlargest(10, "wickets")[["bowler","wickets","average","economy"]].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
sns.barplot(data=bowler_stats.nlargest(15,'wickets'), y='bowler', x='wickets', ax=axes[0], palette='viridis', hue='bowler', legend=False)
axes[0].set_title('Top 15 by Wickets'); axes[0].set_ylabel('')
sns.barplot(data=bowler_stats.nsmallest(15,'economy'), y='bowler', x='economy', ax=axes[1], palette='magma', hue='bowler', legend=False)
axes[1].set_title('Best Economy (min 300)'); axes[1].set_ylabel('')
sns.barplot(data=bowler_stats.nsmallest(15,'average'), y='bowler', x='average', ax=axes[2], palette='plasma', hue='bowler', legend=False)
axes[2].set_title('Best Average (min 300)'); axes[2].set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_top_bowlers.png'), dpi=150, bbox_inches='tight')
plt.show()

season_bowlers = deliveries.groupby(['season_year','bowler']).agg(w=('is_wicket','sum'),balls=('ball','count')).reset_index()
season_bowlers = season_bowlers[season_bowlers['balls']>=100]
top = season_bowlers.loc[season_bowlers.groupby('season_year')['w'].idxmax()]
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=top, x='season_year', y='w', hue='bowler', ax=ax, dodge=False, legend=False)
for _, r in top.iterrows(): ax.text(int(r['season_year']), r['w']+0.3, r['bowler'], ha='center', fontsize=8, rotation=45)
ax.set_title('Highest Wicket-Taker Each Season'); ax.set_xlabel('Season'); ax.set_ylabel('Wickets')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04b_top_bowler_per_season.png'), dpi=150, bbox_inches='tight')
plt.show()

## Bonus: Hidden Patterns

In [ ]:
inn1 = deliveries[deliveries["innings"]==1].groupby("match_id").agg(score1=("total_runs","sum")).reset_index()
inn2 = deliveries[deliveries["innings"]==2].groupby("match_id").agg(score2=("total_runs","sum")).reset_index()
m = inn1.merge(inn2, on="match_id").merge(matches[["match_id","winner"]], on="match_id")
inn2_winners = deliveries[deliveries["innings"]==2].groupby("match_id").agg(bat2=("batting_team","first")).reset_index()
m = m.merge(inn2_winners, on="match_id")
m["chasing_won"] = (m["bat2"] == m["winner"]).astype(int)
chase_win = m["chasing_won"].mean() * 100
print(f"Chasing: {chase_win:.1f}% | Batting first: {100-chase_win:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].pie([chase_win, 100-chase_win], labels=['Chasing Won','Batting First Won'],
            autopct='%1.1f%%', colors=['#1abc9c','#9b59b6'], startangle=90)
axes[0].set_title('Chasing vs Defending')

m2 = m.merge(matches[['match_id','season_year']], on='match_id')
sc = m2.groupby('season_year')['chasing_won'].mean() * 100
axes[1].plot(sc.index.astype(int), sc.values, marker='s', color='#1abc9c', linewidth=2)
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Season'); axes[1].set_ylabel('Chasing Win %'); axes[1].set_title('Chasing Over Seasons')

vs = matches.groupby('venue').agg(mp=('match_id','count'), wr=('toss_winner_won_match','mean')).reset_index()
vs = vs[vs['mp']>=10].sort_values('wr', ascending=False).head(20)
sns.barplot(data=vs, y='venue', x='wr', ax=axes[2], palette='coolwarm', hue='venue', legend=False)
axes[2].set_title('Toss Win % by Venue'); axes[2].set_xlabel('Win %'); axes[2].set_xlim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_hidden_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sr = deliveries.groupby(['season_year','innings']).apply(
    lambda x: (x['total_runs'].sum()/x['ball'].count())*6, include_groups=False
).reset_index()
sr.columns = ['season_year','innings','run_rate']
fig, ax = plt.subplots(figsize=(12, 5))
for i in [1,2]:
    d = sr[sr['innings']==i]
    ax.plot(d['season_year'].astype(int), d['run_rate'], marker='o', label=f'Innings {i}', linewidth=2)
ax.set_xlabel('Season'); ax.set_ylabel('Run Rate'); ax.set_title('IPL Run Rate Evolution'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_run_rate_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Download all results
import shutil
shutil.make_archive("/content/ipl_results", 'zip', OUTPUT_DIR)
from google.colab import files
files.download("/content/ipl_results.zip")

## Summary

| Question | Finding |
|---|---|
| Toss to Win? | 50.6% - marginal |
| Most impactful phase | Death overs (16-20) |
| Top batter | Most runs: Virat Kohli |
| Top bowler | Most wickets: Yuzvendra Chahal |
| Chasing vs Defending | 53.9% chasing wins |
| Run rate trend | Keeps increasing |

## Your Task: ONE Surprising Insight
Write 1-2 paragraphs about something that genuinely surprised you.

## Submission Checklist
- [ ] Add your name at top
- [ ] Interpret each chart
- [ ] Write your surprising insight
- [ ] Run last cell to download ZIP
- [ ] Upload to GitHub
